In [1]:
import numpy as np
from reachy_sdk import ReachySDK
from reachy_sdk.trajectory import goto
from reachy_sdk.trajectory.interpolation import InterpolationMode

reachy = ReachySDK(host="10.22.128.166")

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:


# --------------------------------------------------
# TFLITE / LITERT INTERPRETER
# --------------------------------------------------

try:
    from tflite_runtime.interpreter import Interpreter
    print("Using tflite_runtime")
except ImportError:
    try:
        from ai_edge_litert.interpreter import Interpreter
        print("Using ai_edge_litert")
    except ImportError:
        import tensorflow as tf
        Interpreter = tf.lite.Interpreter
        print("Using TensorFlow Lite")

# --------------------------------------------------
# CONFIGURATION
# --------------------------------------------------

MODEL_PATH = "best.tflite"

CLASS_NAMES = [
    "cube",
    "cylinder",
    "empty"
]

INPUT_SIZE = 320

# Start fairly low for initial testing.
CONF_THRESHOLD = 0.30
NMS_THRESHOLD = 0.45


# --------------------------------------------------
# LOAD MODEL
# --------------------------------------------------

interpreter = Interpreter(model_path=MODEL_PATH)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

input_index = input_details[0]["index"]
output_index = output_details[0]["index"]

print("Model loaded successfully")
print("Input shape :", input_details[0]["shape"])
print("Input dtype :", input_details[0]["dtype"])
print("Output shape:", output_details[0]["shape"])
print("Output dtype:", output_details[0]["dtype"])


# --------------------------------------------------
# YOLO DETECTION
# --------------------------------------------------

def detect(frame):

    original_h, original_w = frame.shape[:2]

    # frame arrives here as BGR.
    # YOLO expects RGB.
    image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Resize to model input size.
    image = cv2.resize(
        image,
        (INPUT_SIZE, INPUT_SIZE)
    )

    # Convert to float32 and normalize 0-1.
    image = image.astype(np.float32) / 255.0

    # HWC -> CHW
    #
    # This is IMPORTANT for your model because its
    # input tensor is [1, 3, 320, 320], not
    # [1, 320, 320, 3].
    image = np.transpose(image, (2, 0, 1))

    # CHW -> NCHW
    image = np.expand_dims(image, axis=0)

    interpreter.set_tensor(
        input_index,
        image
    )

    interpreter.invoke()

    output = interpreter.get_tensor(
        output_index
    )

    # Your output is:
    # [1, 7, 2100]
    #
    # Convert to:
    # [2100, 7]
    predictions = output[0].T

    boxes = []
    confidences = []
    class_ids = []

    for prediction in predictions:

        # YOLO box
        x_center = prediction[0]
        y_center = prediction[1]
        width = prediction[2]
        height = prediction[3]

        # Remaining 3 values are class scores
        class_scores = prediction[4:]

        class_id = int(np.argmax(class_scores))
        confidence = float(class_scores[class_id])

        if confidence < CONF_THRESHOLD:
            continue

        # Convert normalized coordinates to
        # original Reachy camera coordinates.
        x_center *= original_w
        y_center *= original_h
        width *= original_w
        height *= original_h

        x = int(x_center - width / 2)
        y = int(y_center - height / 2)

        w = int(width)
        h = int(height)

        boxes.append([x, y, w, h])
        confidences.append(confidence)
        class_ids.append(class_id)

    # --------------------------------------------------
    # NON-MAXIMUM SUPPRESSION
    # --------------------------------------------------

    detections = []

    if len(boxes) == 0:
        return detections

    indices = cv2.dnn.NMSBoxes(
        boxes,
        confidences,
        CONF_THRESHOLD,
        NMS_THRESHOLD
    )

    for i in indices:

        # Handles different OpenCV return formats.
        if isinstance(i, (list, tuple, np.ndarray)):
            i = int(np.array(i).flatten()[0])
        else:
            i = int(i)

        x, y, w, h = boxes[i]

        detections.append({
            "class_id": class_ids[i],
            "label": CLASS_NAMES[class_ids[i]],
            "confidence": confidences[i],
            "box": (x, y, w, h)
        })

    return detections

# --------------------------------------------------
# BOARD MAPPING
# --------------------------------------------------

BOARD_CENTERS = {
    "A1": (95, 380),
    "A2": (195, 380),
    "A3": (292, 380),

    "B1": (80, 475),
    "B2": (195, 475),
    "B3": (305, 475),

    "C1": (75, 570),
    "C2": (195, 570),
    "C3": (315, 570),
}


def identify_square(box):

    x, y, w, h = box

    # Center of detected object
    cx = x + w / 2
    cy = y + h / 2

    nearest_square = None
    minimum_distance = float("inf")

    for square, (sx, sy) in BOARD_CENTERS.items():

        distance = (
            (cx - sx) ** 2 +
            (cy - sy) ** 2
        ) ** 0.5

        if distance < minimum_distance:
            minimum_distance = distance
            nearest_square = square

    return nearest_square, minimum_distance


Using tflite_runtime
Model loaded successfully
Input shape : [  1   3 320 320]
Input dtype : <class 'numpy.float32'>
Output shape: [   1    7 2100]
Output dtype: <class 'numpy.float32'>


In [3]:
import cv2
import numpy as np

# Capture one frame
frame = reachy.right_camera.wait_for_new_frame()

frame = np.array(frame)

# Convert RGB to BGR
frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)

# Run YOLO
detections = detect(frame)

print("\nBOARD DETECTIONS")
print("-----------------------------")

for detection in detections:

    square, distance = identify_square(
        detection["box"]
    )

    print(
        f"{square}: "
        f"{detection['label']} "
        f"(confidence={detection['confidence']:.2f}, "
        f"distance={distance:.1f})"
    )


BOARD DETECTIONS
-----------------------------


In [4]:
def find_empty_squares():

    frame = reachy.right_camera.wait_for_new_frame()

    frame = np.array(frame)
    frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)

    detections = detect(frame)

    empty_squares = []

    for detection in detections:

        if detection["label"] != "empty":
            continue

        square, distance = identify_square(
            detection["box"]
        )

        # Reject detections too far from a square center
        if distance > 30:
            continue

        if square not in empty_squares:
            empty_squares.append(square)

    return sorted(empty_squares)


# TEST
empty_squares = find_empty_squares()

print("Empty squares:", empty_squares)

Empty squares: []


In [5]:
# Identify available squares
empty_squares = find_empty_squares()

print("Available squares:", empty_squares)

# Our first vision-controlled destination
destination = "B2"

if destination not in empty_squares:
    print("B2 is occupied or could not be confirmed empty.")
    print("Pick-and-place will NOT execute.")
else:
    print("B2 is confirmed empty.")
    print("Destination selected:", destination)
    print("Ready for pick-and-place.")

Available squares: []
B2 is occupied or could not be confirmed empty.
Pick-and-place will NOT execute.


In [ ]:

# ============================================
# PICK AND PLACE
# ============================================

def pick_and_place():

    print("\n=== STARTING PICK AND PLACE ===")

    input("Confirm cube is at its calibrated starting position. Press Enter...")

    # 1. Open
    print("\n1. Opening gripper")
    open_gripper()

    # 2. Approach cube
    print("\n2. Moving to PICK_SAFE")
    move_arm(PICK_SAFE)

    # 3. Lower
    print("\n3. Moving to PICK")
    move_arm(PICK, 5.0)

    # 4. Grasp
    print("\n4. Grasping cube")
    close_gripper()

    input("Confirm cube is securely grasped. Press Enter to lift...")

    # 5. Lift
    print("\n5. Lifting cube")
    move_arm(PICK_SAFE, 5.0)

    input("Confirm cube lifted successfully. Press Enter to transport...")

    # 6. Transport
    print("\n6. Moving to TRANSPORT")
    move_arm(TRANSPORT)

    # 7. Approach destination
    print("\n7. Moving to B2_SAFE")
    move_arm(B2_SAFE)

    input("Confirm alignment above B2. Press Enter to lower...")

    # 8. Lower cube
    print("\n8. Moving to B2_PLACE")
    move_arm(B2_PLACE, 5.0)

    input("Confirm cube is resting on B2. Press Enter to release...")

    # 9. Release
    print("\n9. Releasing cube")
    open_gripper()

    input("Confirm cube is fully released and gripper is clear. Press Enter to retract...")

    # 10. Retract
    print("\n10. Moving to B2_SAFE")
    move_arm(B2_SAFE, 5.0)

    # 11. Return to transport
    print("\n11. Returning to TRANSPORT")
    move_arm(TRANSPORT)

    print("\n=== PICK AND PLACE COMPLETED ===")

In [ ]:
import time

# ============================================
# CONFIGURATION
# ============================================

GRIPPER_OPEN = -29.0
GRIPPER_CLOSE = -10.0

MOVE_DURATION = 6.0
GRIPPER_WAIT = 4.0

# ============================================
# CALIBRATED ARM POSITIONS
# ============================================

PICK_SAFE = {
    reachy.r_arm.r_shoulder_pitch: 14.20,
    reachy.r_arm.r_shoulder_roll: 1.36,
    reachy.r_arm.r_arm_yaw: -16.35,
    reachy.r_arm.r_elbow_pitch: -95.98,
    reachy.r_arm.r_forearm_yaw: -3.16,
    reachy.r_arm.r_wrist_pitch: -17.58,
    reachy.r_arm.r_wrist_roll: 28.48,
}

PICK = {
    reachy.r_arm.r_shoulder_pitch: 14.28,
    reachy.r_arm.r_shoulder_roll: 2.15,
    reachy.r_arm.r_arm_yaw: -15.21,
    reachy.r_arm.r_elbow_pitch: -85.43,
    reachy.r_arm.r_forearm_yaw: -6.68,
    reachy.r_arm.r_wrist_pitch: -13.62,
    reachy.r_arm.r_wrist_roll: 26.10,
}

TRANSPORT = {
    reachy.r_arm.r_shoulder_pitch: 3.56,
    reachy.r_arm.r_shoulder_roll: -2.59,
    reachy.r_arm.r_arm_yaw: 25.22,
    reachy.r_arm.r_forearm_yaw: 0.62,
    reachy.r_arm.r_elbow_pitch: -119.44,
    reachy.r_arm.r_wrist_pitch: 25.58,
    reachy.r_arm.r_wrist_roll: 35.07,
}

B2_SAFE = {
    reachy.r_arm.r_shoulder_pitch: 8.22,
    reachy.r_arm.r_shoulder_roll: -0.39,
    reachy.r_arm.r_arm_yaw: 33.57,
    reachy.r_arm.r_elbow_pitch: -113.64,
    reachy.r_arm.r_forearm_yaw: 1.85,
    reachy.r_arm.r_wrist_pitch: 13.36,
    reachy.r_arm.r_wrist_roll: 30.15,
}

B2_PLACE = {
    reachy.r_arm.r_shoulder_pitch: 7.78,
    reachy.r_arm.r_shoulder_roll: -6.37,
    reachy.r_arm.r_arm_yaw: 35.24,
    reachy.r_arm.r_elbow_pitch: -75.94,
    reachy.r_arm.r_forearm_yaw: -1.76,
    reachy.r_arm.r_wrist_pitch: -25.22,
    reachy.r_arm.r_wrist_roll: 22.50,
}

# ============================================
# HELPER FUNCTIONS
# ============================================

g = reachy.r_arm.r_gripper

def move_arm(pose, duration=MOVE_DURATION):
    goto(
        goal_positions=pose,
        duration=duration,
        interpolation_mode=InterpolationMode.MINIMUM_JERK
    )


def open_gripper():
    g.goal_position = GRIPPER_OPEN
    time.sleep(GRIPPER_WAIT)

    actual = g.present_position
    print(f"Gripper open: {actual:.2f}")

    if abs(actual - GRIPPER_OPEN) > 3.0:
        raise RuntimeError("Gripper failed to open.")


def close_gripper():
    g.goal_position = GRIPPER_CLOSE
    time.sleep(GRIPPER_WAIT)

    print(f"Gripper position: {g.present_position:.2f}")
    print(f"Gripper load: {g.present_load:.2f}")

    # A cube may prevent the gripper from reaching its goal.
    # Therefore, visual confirmation is required.

In [ ]:
def vision_guided_pick_and_place():

    print("\n=== VISION-GUIDED PICK AND PLACE ===")

    # 1. Identify empty squares
    empty_squares = find_empty_squares()

    print("Empty squares:", empty_squares)

    # 2. Verify B2 is available
    destination = "B2"

    if destination not in empty_squares:
        print("STOP: B2 is not confirmed empty.")
        return

    print("B2 confirmed empty by YOLO.")

    # 3. Request confirmation before moving
    confirmation = input(
        "Cube at pickup position and path clear? (yes/no): "
    )

    if confirmation.lower() != "yes":
        print("Operation cancelled.")
        return

    # 4. Execute our tested Phase A program
    pick_and_place()

    print("\nVision-guided placement completed!")


# Execute
vision_guided_pick_and_place()

In [ ]:
reachy.r_arm.compliant = True

In [ ]:
reachy.turn_off("r_arm")

In [ ]:
print("RIGHT ARM MOTOR STATUS")
print("-----------------------------")

for joint in reachy.r_arm.joints.values():
    print(
        f"{joint.name:22s} "
        f"Compliant: {joint.compliant} "
        f"Position: {joint.present_position:.2f}"
    )

In [ ]:
def record_arm_pose():
    return {
        joint: joint.present_position
        for joint in reachy.r_arm.joints.values()
        if joint.name != "r_gripper"
    }

B3_SAFE = record_arm_pose()

print("\nB3_SAFE = {")
for joint, angle in B3_SAFE.items():
    print(f"    reachy.r_arm.{joint.name}: {angle:.2f},")
print("}")

In [ ]:
B3_PLACE = {
    joint: joint.present_position
    for joint in reachy.r_arm.joints.values()
    if joint.name != "r_gripper"
}

print("\nB3_PLACE = {")
for joint, angle in B3_PLACE.items():
    print(f"    reachy.r_arm.{joint.name}: {angle:.2f},")
print("}")

In [ ]:
print("Checking live joint feedback...")

joint = reachy.r_arm.r_elbow_pitch

print("Current reading:", joint.present_position)

input("Gently move the elbow joint, keeping the arm supported. Press Enter...")

print("Reading after movement:", joint.present_position)